# 실습 1: MNIST 파이프라인 Overview (데이터부터 학습까지 한 번에)

이 실습은 모델 훈련을 위해 데이터가 어떻게 준비되고, 모델에 들어가며, 결과가 나오는지를 '처음부터 끝까지' 한 번에 체험하는 오버뷰(Overview) 실습입니다.

**개념 복기 및 이론 점검**
- PyTorch의 `Dataset`과 `DataLoader`가 왜 분리되어 있는지 생각해 봅니다.
- 이미지를 딥러닝 모델에 넣기 전에 어떤 변환(`transforms`)이 필요한지 관찰합니다.
- 훈련(Train) 데이터와 평가(Test) 데이터를 나누는 이유를 상기합니다.

[!Open In Colab](https://colab.research.google.com/github/Team-AnI/A-AND-I-4TH-AI-CODE-LAB/blob/main/3주차/lab_01_pipeline.ipynb)

> 위 배지를 누르면 이 노트북이 **여러분 Google 계정의 Colab**에서 열립니다. 수정본을 남기려면 `파일 → Drive에 사본 저장`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

# 재현성을 위한 시드 고정
torch.manual_seed(42)

# 디바이스 설정 (cuda -> mps -> cpu)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 데이터 준비 및 전처리
딥러닝 모델은 숫자로 된 텐서만 이해합니다. 이미지를 텐서로 바꾸고(`ToTensor`), 학습이 잘 되도록 값의 범위를 조정(`Normalize`)합니다.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(dataset=train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

## 2. 데이터 시각화
데이터가 어떻게 생겼는지 눈으로 확인합니다. 모델 학습 전 데이터를 뜯어보는 것은 필수적인 습관입니다.

In [ ]:
dataiter = iter(train_loader)
images, labels = next(dataiter)

fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i in range(4):
    # 정규화된 텐서를 시각화를 위해 대략적인 범위 복구
    img = images[i].squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"Label: {labels[i].item()}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 3. 🔑 핵심 실습: MLP 모델 정의
28x28 크기의 2차원 이미지를 1차원으로 쫙 펴서(Flatten) 다층 퍼셉트론(MLP)에 입력합니다.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super(SimpleMLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 128)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

model = SimpleMLP().to(device)
print(model)

## 4. 모델 훈련 및 평가 파이프라인
손실 함수(Loss)와 최적화 도구(Optimizer)를 정의하고, 훈련 루프를 돌며 모델을 학습시킵니다. 여기서는 파이프라인이 끝까지 돌아가는 모습을 관찰하는 것이 핵심입니다.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 3
train_losses = []

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    # tqdm으로 진행률 및 실시간 Loss 표시
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for batch_idx, (data, target) in enumerate(pbar):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = running_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{epochs}] Average Loss: {avg_loss:.4f}")
    
    # 평가
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
    print(f"Test Accuracy: {100 * correct / total:.2f}%")

## 5. 학습 결과 시각화
손실(Loss)이 잘 떨어졌는지 시각화합니다.

In [ ]:
plt.plot(range(1, epochs + 1), train_losses, marker='o')
plt.title('Training Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(range(1, epochs + 1))
plt.grid(True)
plt.show()

## 6. 예측 결과 시각화
모델이 어떤 이미지를 맞고 틀렸는지 직접 확인합니다. 모델의 강점과 약점을 파악하는 정성적인 분석 단계입니다.

In [ ]:
# 모델을 평가 모드로 설정하고 예측 수행
model.eval()
correct_examples = []
incorrect_examples = []

with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        _, predicted = torch.max(output.data, 1)
        
        corrects = (predicted == target)
        # 맞은/틀린 샘플을 충분히 찾을 때까지 반복
        for i in range(len(corrects)):
            if corrects[i] and len(correct_examples) < 5:
                correct_examples.append({
                    'image': data[i].cpu(),
                    'true_label': target[i].cpu(),
                    'pred_label': predicted[i].cpu()
                })
            elif not corrects[i] and len(incorrect_examples) < 5:
                 incorrect_examples.append({
                    'image': data[i].cpu(),
                    'true_label': target[i].cpu(),
                    'pred_label': predicted[i].cpu()
                })
        
        if len(correct_examples) >= 5 and len(incorrect_examples) >= 5:
            break

# 맞은 예측 시각화
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
fig.suptitle('Correct Predictions', fontsize=16)
for i, ex in enumerate(correct_examples):
    img = ex['image'].squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"True: {ex['true_label']}\nPred: {ex['pred_label']}")
    axes[i].axis('off')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# 틀린 예측 시각화
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
fig.suptitle('Incorrect Predictions', fontsize=16)
for i, ex in enumerate(incorrect_examples):
    img = ex['image'].squeeze().numpy()
    axes[i].imshow(img, cmap='gray')
    axes[i].set_title(f"True: {ex['true_label']}\nPred: {ex['pred_label']}")
    axes[i].axis('off')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 7. ✅ 학습 결과 정리
- 데이터셋이 `DataLoader`를 거쳐 배치 단위로 제공되는 과정을 확인했습니다.
- 전처리(`transforms`)된 데이터가 MLP 모델을 거쳐 예측값을 뱉어내고, Loss에 따라 최적화되는 한 사이클을 직접 실행했습니다.
- 🎯 **핵심 결론:** 코드가 어떻게 생겼는지 눈에 익혔다면 성공입니다! 다음 실습(lab_02)에서는 이 코드들을 **한 줄씩 분해하여 왜 그렇게 짜여 있는지** 깊게 알아봅니다.